# DocX↔PDF Converter — Backend

1. Rode as 3 células abaixo em ordem.
2. Copie a URL `https://xxxx.trycloudflare.com` que aparece na saída.
3. No site [DocX Converter](https://xangrybadger.github.io/docx-pdf-converter/), clique em **Sem API** no header e cole a URL.

A URL muda a cada sessão — você precisa colar novamente se reiniciar o notebook.

In [ ]:
!pip install -q fastapi uvicorn python-docx reportlab openpyxl pypdf python-multipart Pillow

In [ ]:
import os
os.makedirs('app', exist_ok=True)

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import os
import uuid
from pathlib import Path
from docx import Document
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from openpyxl import load_workbook
from pypdf import PdfReader, PdfWriter

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

TEMP_DIR = Path("/tmp/docx-converter")
TEMP_DIR.mkdir(exist_ok=True)

def convert_docx_to_pdf(docx_path: str, output_path: str):
    doc = Document(docx_path)
    doc_pdf = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    for para in doc.paragraphs:
        story.append(Paragraph(para.text, styles['Normal']))
        story.append(Spacer(1, 6))

    doc_pdf.build(story)

def convert_xlsx_to_pdf(xlsx_path: str, output_path: str):
    wb = load_workbook(xlsx_path, data_only=True)
    doc_pdf = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    for sheet in wb.worksheets:
        story.append(Paragraph(f"Sheet: {sheet.title}", styles['Heading1']))
        story.append(Spacer(1, 6))

        max_row = sheet.max_row or 0
        max_col = sheet.max_column or 0

        for row in sheet.iter_rows(min_row=1, max_row=min(50, max_row), min_col=1, max_col=min(20, max_col)):
            row_text = " | ".join(str(cell.value) if cell.value else "" for cell in row)
            story.append(Paragraph(row_text, styles['Code']))
            story.append(Spacer(1, 3))

        story.append(Spacer(1, 12))

    doc_pdf.build(story)

def compress_pdf_file(input_path: str, output_path: str):
    reader = PdfReader(input_path)
    writer = PdfWriter()

    for page in reader.pages:
        writer.add_page(page)

    with open(output_path, "wb") as f:
        writer.write(f)

@app.post("/api/convert/docx-pdf")
async def convert_docx_pdf(file: UploadFile = File(...)):
    try:
        uid = uuid.uuid4().hex[:8]
        temp_input = TEMP_DIR / f"input_{uid}_{file.filename}"
        temp_output = TEMP_DIR / f"output_{uid}_{file.filename.replace('.docx', '.pdf')}"

        with open(temp_input, "wb") as f:
            content = await file.read()
            f.write(content)

        convert_docx_to_pdf(str(temp_input), str(temp_output))

        return FileResponse(
            str(temp_output),
            media_type="application/pdf",
            filename=temp_output.name
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/convert/xlsx-pdf")
async def convert_xlsx_pdf(file: UploadFile = File(...)):
    try:
        uid = uuid.uuid4().hex[:8]
        temp_input = TEMP_DIR / f"input_{uid}_{file.filename}"
        temp_output = TEMP_DIR / f"output_{uid}_{file.filename.replace('.xlsx', '.pdf')}"

        with open(temp_input, "wb") as f:
            content = await file.read()
            f.write(content)

        convert_xlsx_to_pdf(str(temp_input), str(temp_output))

        return FileResponse(
            str(temp_output),
            media_type="application/pdf",
            filename=temp_output.name
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/convert/pdf-compress")
async def compress_pdf(file: UploadFile = File(...)):
    try:
        uid = uuid.uuid4().hex[:8]
        temp_input = TEMP_DIR / f"input_{uid}_{file.filename}"
        temp_output = TEMP_DIR / f"compressed_{uid}_{file.filename}"

        with open(temp_input, "wb") as f:
            content = await file.read()
            f.write(content)

        compress_pdf_file(str(temp_input), str(temp_output))

        return FileResponse(
            str(temp_output),
            media_type="application/pdf",
            filename=temp_output.name
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/convert/batch")
async def convert_batch(files: list[UploadFile] = File(...)):
    try:
        results = []
        for file in files:
            try:
                uid = uuid.uuid4().hex[:8]
                temp_input = TEMP_DIR / f"input_{uid}_{file.filename}"

                with open(temp_input, "wb") as f:
                    content = await file.read()
                    f.write(content)

                if file.filename.endswith('.docx'):
                    temp_output = TEMP_DIR / f"output_{uid}_{file.filename.replace('.docx', '.pdf')}"
                    convert_docx_to_pdf(str(temp_input), str(temp_output))
                elif file.filename.endswith('.xlsx'):
                    temp_output = TEMP_DIR / f"output_{uid}_{file.filename.replace('.xlsx', '.pdf')}"
                    convert_xlsx_to_pdf(str(temp_input), str(temp_output))
                else:
                    results.append({
                        "original": file.filename,
                        "status": "error",
                        "error": "Formato não suportado no modo lote. Use DOCX ou XLSX."
                    })
                    continue

                results.append({
                    "original": file.filename,
                    "converted": temp_output.name,
                    "status": "success"
                })
            except Exception as e:
                results.append({
                    "original": file.filename,
                    "status": "error",
                    "error": str(e)
                })

        return {"results": results}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "DocX-PDF Converter API"}

In [ ]:
import subprocess, threading, time, re, urllib.request

!command -v cloudflared >/dev/null 2>&1 || curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

def run():
    import uvicorn
    uvicorn.run('app.main:app', host='0.0.0.0', port=8000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

url = None
for line in proc.stdout:
    decoded = line.decode(errors='replace')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', decoded)
    if m:
        url = m.group(0)
        break

if url:
    print(f'\n{"="*60}')
    print(f' DocX Converter Backend Online!')
    print(f' URL: {url}')
    print(f'{"="*60}')
    print(f'\n Cole esta URL no header do site:')
    print(f' https://xangrybadger.github.io/docx-pdf-converter/')
    print()
else:
    print('Aguardando URL do cloudflared...')
    print('Verifique a saída acima.')